# Extract Json Data

Process to extract json data:
1. Inspect url: https://www.theguardian.com/education/ng-interactive/2025/sep/13/the-guardian-university-guide-2026-the-rankings
2. Look under Networks
3. Re-fresh page and search under Fetch/XHR for json file
4. Run the code below using json url extracted from step 3

In [8]:
import pandas as pd
import requests

url = "https://interactive.guim.co.uk/atoms/labs/2025/09/university-guide/overview/v/1771858244880/assets/data/overview.json"
response = requests.get(url, timeout=30)
response.raise_for_status()
data = response.json()

universities = pd.DataFrame(data["institutions"])
universities

,institutionId,name,url,rank2026,rank2025,rank2024,guardianScore,percentSatisfiedWithTeaching,expenditurePerStudent,studentStaffRatio,careerProspects,valueAdded,averageEntryTariff,percentSatisfiedWithAssessment,continuation
0,0156,Oxford,https://www.theguardian.com/education/2009/may...,1,1.0,2.0,100,94,10,9.1,92,6.3,197,76,97.3
1,0173,St Andrews,https://www.theguardian.com/education/2009/may...,2,2.0,1.0,98.4,92.8,6.5,11,90,5.5,210,81.7,96.8
2,0114,Cambridge,https://www.theguardian.com/education/2009/may...,3,3.0,3.0,94.6,93.4,9.4,10.8,92,5.9,210,72.9,98.4
3,0137,London School of Economics,https://www.theguardian.com/education/2009/may...,4,4.0,4.0,93.8,89.4,8.7,11.4,93,5.9,185,76.2,96.4
4,0116,Durham,https://www.theguardian.com/education/2009/may...,5,6.0,7.0,85.5,90.5,6.4,12.4,89,6,180,81.6,96.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,0059,Greenwich,https://www.theguardian.com/education/2009/may...,119,117.0,116.0,38.5,84.6,3.5,16.9,75,5,108,82.2,84.6
119,0031,Roehampton,https://www.theguardian.com/education/2009/may...,120,110.0,111.0,36.3,84.6,4.8,20.9,67,3.9,94,82.8,87.5
120,0067,Middlesex,https://www.theguardian.com/education/2009/may...,121,113.0,115.0,36.1,87.1,None,17.4,69,5.9,103,84.3,78.8
121,0083,Westminster,https://www.theguardian.com/education/2009/may...,122,119.0,117.0,34.2,83.5,4.3,19.9,69,4.5,110,81.9,86.5


## Extract Subject Areas
From data extract subject areas containing individual rankings for each area.

In [12]:
# Extract subject IDs and names for downloading individual subject rankings
subjects_data = [
    {
        "id": subject["id"],
        "title": subject["title"],
    }
    for subject in data.get("subjects", [])
    if subject.get("id") and subject.get("title")
]

subjects_data

[{'id': 'S250', 'title': 'Accounting and finance'},
 {'id': 'S195', 'title': 'Aerospace engineering'},
 {'id': 'S040', 'title': 'Anatomy and physiology'},
 {'id': 'S150', 'title': 'Animal science and agriculture'},
 {'id': 'S225', 'title': 'Animation and game design'},
 {'id': 'S325', 'title': 'Anthropology and archaeology'},
 {'id': 'S450', 'title': 'Architecture'},
 {'id': 'S100', 'title': 'Biology'},
 {'id': 'S045', 'title': 'Biomedical science'},
 {'id': 'S240', 'title': 'Business and management'},
 {'id': 'S170', 'title': 'Chemical engineering'},
 {'id': 'S110', 'title': 'Chemistry'},
 {'id': 'S053', 'title': "Children's nursing"},
 {'id': 'S200', 'title': 'Civil engineering'},
 {'id': 'S460', 'title': 'Classics and ancient history'},
 {'id': 'S220', 'title': 'Computer science and information systems'},
 {'id': 'S500', 'title': 'Construction, surveying and planning'},
 {'id': 'S385', 'title': 'Creative writing'},
 {'id': 'S315', 'title': 'Criminology'},
 {'id': 'S020', 'title': 'D

### Extract json data for individual subject areas

In [16]:
subject_dataframes = {}

base_url = "https://interactive.guim.co.uk/atoms/labs/2025/09/university-guide/overview/v/1771858244880/assets/data"

for subject in subjects_data:
    
    subject_id = subject["id"]
    subject_name = subject["title"]
    subject_url = f"{base_url}/{subject_id}.json"

    subject_response = requests.get(subject_url, timeout=30)
    subject_response.raise_for_status()
    subject_json = subject_response.json()

    records = subject_json.get("institutions", [])

    subject_dataframes[subject_id] = (
        pd.json_normalize(records)
        .assign(subject_id=subject_id, subject=subject_name)
    )

# Example: the DataFrame for subject S102
subject_dataframes["S102"]

,institutionId,name,url,courses,rank,guardianScore,percentSatisfiedWithTeaching,expenditurePerStudent,studentStaffRatio,careerProspects,valueAdded,averageEntryTariff,percentSatisfiedWithAssessment,continuation,subject_id,subject
0,0168,Glasgow,https://www.theguardian.com/education/2009/may...,[{'url': 'http://www.gla.ac.uk/undergraduate/d...,1,100,88.2,9.0,14.5,75.0,8.0,216,81.8,92.5,S102,Zoology
1,0112,Bristol,https://www.theguardian.com/education/2009/may...,[{'url': 'https://www.bristol.ac.uk/study/unde...,2,94.8,96.3,8.0,14.6,68.0,5.0,156,77,100,S102,Zoology
2,0105,West of Scotland,https://www.theguardian.com/education/2009/may...,[{'url': 'https://www.uws.ac.uk/study/undergra...,3,92.9,99.1,3.0,12,NaN,8.0,141,95.9,None,S102,Zoology
3,0180,Swansea,https://www.theguardian.com/education/2009/may...,[{'url': 'https://www.swansea.ac.uk/undergradu...,4,91,96.1,7.0,12.8,71.0,5.0,133,86.3,94.9,S102,Zoology
4,0119,Exeter,https://www.theguardian.com/education/2009/may...,[{'url': 'https://www.exeter.ac.uk/study/under...,5,83.8,84.5,6.0,12.3,71.0,8.0,151,70.7,93.2,S102,Zoology
5,0126,Liverpool,https://www.theguardian.com/education/2009/may...,[{'url': 'https://www.liverpool.ac.uk/courses/...,6,81.4,83.3,NaN,9,NaN,6.0,145,75.8,95.7,S102,Zoology
6,0066,Manchester Met,https://www.theguardian.com/education/2009/may...,[],7,81.3,None,4.0,15.6,NaN,8.0,129,None,91.5,S102,Zoology
7,0204,Manchester,https://www.theguardian.com/education/2009/may...,[{'url': 'https://www.manchester.ac.uk/study/u...,8,81.1,88.7,10.0,17.2,NaN,8.0,174,64.7,100,S102,Zoology
8,0062,Lincoln,https://www.theguardian.com/education/2009/may...,[{'url': 'https://www.lincoln.ac.uk/home/cours...,9,78,87.4,3.0,15.4,56.0,7.0,115,88.5,97.5,S102,Zoology
9,0155,Nottingham,https://www.theguardian.com/education/2009/may...,[{'url': 'https://www.nottingham.ac.uk/studywi...,10,76.9,80,9.0,12.1,NaN,6.0,150,74.4,94.1,S102,Zoology
